# SOH Forecasting for Li-ion EV Battery (`#14.csv`)

Cycle-level Transformer-encoder approach.

**Pipeline**

1. Load and parse charging data
2. Segment into charging cycles (gap-based)
3. Aggregate per-cycle features and derive a smoothed SOH proxy from deep-charge cycles
4. Build sliding windows of past cycles to forecast the next cycle's SOH
5. Train a small Transformer encoder
6. Evaluate against persistence and linear baselines on a held-out tail of cycles

> Requires: `pandas numpy matplotlib scikit-learn torch`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.manual_seed(0)
np.random.seed(0)

CSV_PATH = "#14.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Load and parse

In [ ]:
RENAME = {
    "pack_voltage (V)":      "pack_voltage",
    "charge_current (A)":    "charge_current",
    "max_cell_voltage (V)":  "max_cell_voltage",
    "min_cell_voltage (V)":  "min_cell_voltage",
    "max_temperature (℃)":     "max_temperature",
    "min_temperature (℃)":     "min_temperature",
    "available_energy (kw)": "available_energy",
    "available_capacity (Ah)": "available_capacity",
}

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={df.columns[0]: "row_idx", **RENAME})
missing = [v for v in RENAME.values() if v not in df.columns]
assert not missing, f"Expected columns missing after rename: {missing}. Got: {list(df.columns)}"
df["timestamp"] = pd.to_datetime(df["record_time"], format="%Y%m%d%H%M%S")
df = df.sort_values("timestamp").reset_index(drop=True)
print("rows:", len(df))
print("columns:", list(df.columns))
print("range:", df["timestamp"].min(), "->", df["timestamp"].max())
df.head()


## 2. Segment charging cycles

A new session starts whenever the gap to the previous sample exceeds `GAP_MINUTES`. We keep a session as a charging cycle only if the median current is negative (the dataset's sign convention) **and** SOC rises by at least `MIN_DELTA_SOC` percentage points.


In [ ]:
GAP_MINUTES = 10
MIN_DELTA_SOC = 5.0

dt = df["timestamp"].diff().dt.total_seconds().fillna(0)
df["session"] = (dt > GAP_MINUTES * 60).cumsum()

plt.figure(figsize=(7, 3))
plt.hist(np.log10(dt[dt > 0] + 1), bins=80)
plt.xlabel("log10(seconds + 1) between consecutive samples")
plt.ylabel("count"); plt.title("Sampling-gap distribution")
plt.show()
print("Distinct sessions:", df["session"].nunique())


In [ ]:
def aggregate_session(g):
    soc_start = g["soc"].iloc[0]; soc_end = g["soc"].iloc[-1]
    cur = g["charge_current"]
    if cur.median() >= -1.0:               # not actually charging
        return None
    if (soc_end - soc_start) < MIN_DELTA_SOC:
        return None
    duration_s = (g["timestamp"].iloc[-1] - g["timestamp"].iloc[0]).total_seconds()
    return pd.Series({
        "t_start":              g["timestamp"].iloc[0],
        "t_end":                g["timestamp"].iloc[-1],
        "duration_min":         duration_s / 60.0,
        "soc_start":            soc_start,
        "soc_end":              soc_end,
        "delta_soc":            soc_end - soc_start,
        "v_pack_mean":          g["pack_voltage"].mean(),
        "v_pack_max":           g["pack_voltage"].max(),
        "v_pack_min":           g["pack_voltage"].min(),
        "i_chg_mean_abs":       cur.abs().mean(),
        "i_chg_max_abs":        cur.abs().max(),
        "temp_max_mean":        g["max_temperature"].mean(),
        "temp_min_mean":        g["min_temperature"].mean(),
        "temp_spread_mean":     (g["max_temperature"] - g["min_temperature"]).mean(),
        "cell_v_max_mean":      g["max_cell_voltage"].mean(),
        "cell_v_spread_mean":   (g["max_cell_voltage"] - g["min_cell_voltage"]).mean(),
        "cap_charged_ah":       g["available_capacity"].iloc[-1] - g["available_capacity"].iloc[0],
        "energy_charged_kwh":   g["available_energy"].iloc[-1]   - g["available_energy"].iloc[0],
        "available_capacity_max": g["available_capacity"].max(),
    })

records = []
for sid, g in df.groupby("session", sort=True):
    rec = aggregate_session(g)
    if rec is not None:
        rec["session"] = sid
        records.append(rec)

cycles = pd.DataFrame(records).reset_index(drop=True)
cycles["cycle_idx"] = np.arange(len(cycles))
cycles["days_since_start"] = (cycles["t_start"] - cycles["t_start"].iloc[0]).dt.total_seconds() / 86400.0
print(f"Detected {len(cycles)} charging cycles over {cycles['days_since_start'].iloc[-1]:.1f} days")
cycles.head()


## 3. Derive SOH

For cycles that reach near-full charge (`soc_end >= DEEP_SOC`), the maximum `available_capacity` reached approximates the full pack capacity at that moment in time. We normalise by an early-life reference, smooth with a rolling median, then linearly interpolate so every cycle has a label.


In [ ]:
DEEP_SOC = 95.0
ROLL_WINDOW = 5
REF_HEAD = 5

deep = cycles[cycles["soc_end"] >= DEEP_SOC].copy().reset_index(drop=True)
print(f"{len(deep)} deep cycles out of {len(cycles)}")

ref_cap = deep["available_capacity_max"].iloc[:REF_HEAD].mean()
deep["soh_raw"]    = deep["available_capacity_max"] / ref_cap
deep["soh_smooth"] = deep["soh_raw"].rolling(ROLL_WINDOW, min_periods=1, center=True).median()

cycles["soh"] = np.interp(cycles["cycle_idx"], deep["cycle_idx"], deep["soh_smooth"])

plt.figure(figsize=(9, 4))
plt.scatter(deep["cycle_idx"], deep["soh_raw"], s=8, alpha=0.4, label="raw SOH (deep cycles)")
plt.plot(cycles["cycle_idx"], cycles["soh"], color="C1", label="smoothed + interpolated SOH")
plt.xlabel("cycle index"); plt.ylabel("SOH"); plt.legend(); plt.title("Derived SOH")
plt.show()


## 4. Build windows and temporal split

In [ ]:
FEATURE_COLS = [
    "duration_min", "delta_soc", "soc_start", "soc_end",
    "v_pack_mean", "v_pack_max", "v_pack_min",
    "i_chg_mean_abs", "i_chg_max_abs",
    "temp_max_mean", "temp_min_mean", "temp_spread_mean",
    "cell_v_max_mean", "cell_v_spread_mean",
    "cap_charged_ah", "energy_charged_kwh",
    "available_capacity_max",
    "days_since_start",
    "soh",  # current SOH is also an input feature for the next-cycle forecast
]
WINDOW = 30      # past cycles fed to the encoder
HORIZON = 1      # forecast next cycle's SOH (single-step)
TRAIN_FRAC = 0.75

X_all = cycles[FEATURE_COLS].values.astype(np.float32)
y_all = cycles["soh"].values.astype(np.float32)

split = int(len(cycles) * TRAIN_FRAC)
print(f"Train cycles 0..{split-1}  |  Test cycles {split}..{len(cycles)-1}")

scaler = StandardScaler().fit(X_all[:split])
X_scaled = scaler.transform(X_all).astype(np.float32)

def make_windows(X, y, window, horizon, start, end):
    Xs, ys = [], []
    for i in range(start, end - window - horizon + 1):
        Xs.append(X[i:i+window])
        ys.append(y[i + window + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(X_scaled, y_all, WINDOW, HORIZON, 0, split)
X_test,  y_test  = make_windows(X_scaled, y_all, WINDOW, HORIZON, split - WINDOW, len(cycles))
print("Train windows:", X_train.shape, "Test windows:", X_test.shape)


In [ ]:
class CycleWindowDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

# Carve a small chronological validation slice off the END of train
val_n = max(32, int(0.1 * len(X_train)))
train_ds = CycleWindowDS(X_train[:-val_n], y_train[:-val_n])
val_ds   = CycleWindowDS(X_train[-val_n:], y_train[-val_n:])
test_ds  = CycleWindowDS(X_test, y_test)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=128)
test_dl  = DataLoader(test_ds,  batch_size=128)
print("train", len(train_ds), "val", len(val_ds), "test", len(test_ds))


## 5. Transformer encoder model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class SOHTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=3, dim_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
    def forward(self, x):
        h = self.pos(self.input_proj(x))
        h = self.encoder(h)
        return self.head(h[:, -1]).squeeze(-1)

model = SOHTransformer(n_features=len(FEATURE_COLS)).to(DEVICE)
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Train

In [ ]:
EPOCHS = 80
LR = 1e-3
PATIENCE = 12

opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.MSELoss()

best_val = float("inf"); best_state = None; bad = 0
hist = {"train": [], "val": []}

for epoch in range(EPOCHS):
    model.train(); tr = 0.0; n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr += loss.item() * len(xb); n += len(xb)
    sched.step(); tr /= n

    model.eval(); vl = 0.0; nv = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += loss_fn(model(xb), yb).item() * len(xb); nv += len(xb)
    vl /= nv
    hist["train"].append(tr); hist["val"].append(vl)

    if vl < best_val - 1e-6:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train {tr:.5f}  val {vl:.5f}  best {best_val:.5f}")
    if bad >= PATIENCE:
        print(f"Early stop at epoch {epoch}, best val {best_val:.5f}")
        break

model.load_state_dict(best_state)

plt.figure(figsize=(7, 3))
plt.plot(hist["train"], label="train")
plt.plot(hist["val"],   label="val")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Loss")
plt.show()


## 7. Evaluate

Compare the Transformer against two cheap baselines:

- **Persistence** — predict next cycle's SOH = current cycle's SOH.
- **Linear extrapolation** — least-squares linear fit of SOH vs cycle index on training cycles, applied to test cycles.


In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval()
    out = []
    for xb, _ in dl:
        out.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

y_pred = predict(model, test_dl)

# Persistence: last SOH in each test window (undo standardisation)
soh_idx = FEATURE_COLS.index("soh")
mu, sg = scaler.mean_[soh_idx], scaler.scale_[soh_idx]
y_persist = X_test[:, -1, soh_idx] * sg + mu

# Linear extrapolation fit on training cycles
train_idx = np.arange(split)
coef = np.polyfit(train_idx, y_all[:split], 1)
test_cycle_idx = np.arange(split, len(cycles))[-len(y_test):]
y_linear = np.polyval(coef, test_cycle_idx)

def report(name, yt, yp):
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    print(f"{name:14s}  MAE {mean_absolute_error(yt, yp):.5f}  RMSE {rmse:.5f}  R2 {r2_score(yt, yp):.4f}")

report("Transformer", y_test, y_pred)
report("Persistence", y_test, y_persist)
report("Linear",      y_test, y_linear)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cycles["cycle_idx"], cycles["soh"], color="lightgray", label="full SOH series")
plt.plot(test_cycle_idx, y_test,    "k",     lw=2, label="test true")
plt.plot(test_cycle_idx, y_pred,    "C0",    label="Transformer")
plt.plot(test_cycle_idx, y_linear,  "C2--",  label="Linear")
plt.plot(test_cycle_idx, y_persist, "C3:",   label="Persistence")
plt.axvline(split, color="red", ls="--", lw=1, label="train/test split")
plt.xlabel("cycle index"); plt.ylabel("SOH")
plt.title("SOH forecast on held-out cycles"); plt.legend()
plt.show()


## Notes & next steps

- **Few cycles?** If `Detected N charging cycles` is small (<200), shrink `WINDOW` to ~10 and reduce `d_model`/`layers`.
- **Noisy SOH?** Increase `ROLL_WINDOW` or tighten `DEEP_SOC` to 97–98.
- **Multi-step forecasting:** change the head to output `HORIZON` values, set `HORIZON=20+`, and update `make_windows` to return a horizon vector. Train with the same MSE.
- **Sanity check the proxy:** `available_capacity_max` is the BMS estimate. As an alternative, set `soh = cap_charged_ah / delta_soc * 100 / ref_full_cap` per cycle (physics-based) and re-run.
- **Sample-level ICA features:** if you want richer per-cycle inputs, compute dQ/dV peak height/position and add them to `FEATURE_COLS`.
